# 01 – Data Cleaning
## Reel Insights: Global Movie Industry (TMDB)

Load six source CSV files, inspect quality, remove duplicates, handle missing values, validate relationships and save cleaned datasets.



## 1. Imports and paths

In [ ]:
import os, pandas as pd, numpy as np
PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CLEANED_DIR = os.path.join(DATA_DIR, "cleaned")
os.makedirs(CLEANED_DIR, exist_ok=True)

## 2. Load source datasets

In [ ]:
files = ["movies", "genres", "movie_genres", "cast", "crew", "movie_keywords"]
data = {}
for name in files:
    path = os.path.join(DATA_DIR, name + ".csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {path}. Add all six original CSV files to data/.")
    data[name] = pd.read_csv(path)
for name, df in data.items():
    print(name, df.shape)
    display(df.head(3))

## 3. Inspect missing values, data types and duplicates

In [ ]:
for name, df in data.items():
    print("\\n", name, "dtypes:")
    print(df.dtypes)
    print("Missing values:")
    display(df.isna().sum().to_frame("missing_count"))
    print("Exact duplicate rows:", df.duplicated().sum())

print("Duplicate movie IDs:")
display(data["movies"][data["movies"].duplicated("movie_id", keep=False)].sort_values("movie_id"))

## 4. Remove duplicates using appropriate keys

In [ ]:
movies = data["movies"].drop_duplicates("movie_id", keep="first").copy()
genres = data["genres"].drop_duplicates("genre_id", keep="first").copy()
movie_genres = data["movie_genres"].drop_duplicates(["movie_id", "genre_id"], keep="first").copy()
cast = data["cast"].drop_duplicates(
    ["movie_id", "person_id", "actor_name", "character_name", "cast_order"], keep="first"
).copy()
crew = data["crew"].drop_duplicates(
    ["movie_id", "person_id", "person_name", "job", "department"], keep="first"
).copy()
movie_keywords = data["movie_keywords"].drop_duplicates(
    ["movie_id", "keyword_id"], keep="first"
).copy()

cleaned = {
    "movies": movies, "genres": genres, "movie_genres": movie_genres,
    "cast": cast, "crew": crew, "movie_keywords": movie_keywords
}
for name, df in cleaned.items():
    print(name, "source rows:", len(data[name]), "cleaned rows:", len(df))

## 5. Handle missing values and convert data types

In [ ]:
cast["character_name"] = cast["character_name"].fillna("Unknown")

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
for col in ["runtime", "budget", "revenue", "popularity", "vote_average", "vote_count"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

print("Invalid or missing release dates:", movies["release_date"].isna().sum())

## 6. Investigate zero budget and revenue values

In [ ]:
zero_budget = movies["budget"].eq(0)
zero_revenue = movies["revenue"].eq(0)
print("Zero budget:", int(zero_budget.sum()))
print("Zero revenue:", int(zero_revenue.sum()))
print("Both zero:", int((zero_budget & zero_revenue).sum()))
print("Both positive:", int(((movies["budget"] > 0) & (movies["revenue"] > 0)).sum()))

# Zero financial values are retained, but flagged as unavailable for financial analysis.
movies["financial_data_available"] = (movies["budget"] > 0) & (movies["revenue"] > 0)
cleaned["movies"] = movies

## 7. Validate keys and relationships

In [ ]:
movie_ids, genre_ids = set(movies.movie_id), set(genres.genre_id)
checks = {
    "movie_genres -> movies": (~movie_genres.movie_id.isin(movie_ids)).sum(),
    "movie_genres -> genres": (~movie_genres.genre_id.isin(genre_ids)).sum(),
    "cast -> movies": (~cast.movie_id.isin(movie_ids)).sum(),
    "crew -> movies": (~crew.movie_id.isin(movie_ids)).sum(),
    "movie_keywords -> movies": (~movie_keywords.movie_id.isin(movie_ids)).sum()
}
print("Unique movie IDs:", movies.movie_id.is_unique)
print("Unique genre IDs:", genres.genre_id.is_unique)
for name, count in checks.items():
    print(name, "orphan rows:", int(count))
for name, df in cleaned.items():
    print(name, "remaining exact duplicates:", df.duplicated().sum())

## 8. Save cleaned CSV files

In [ ]:
for name, df in cleaned.items():
    output = os.path.join(CLEANED_DIR, f"{name}_cleaned.csv")
    df.to_csv(output, index=False)
    print("Saved", output, df.shape)

## 9. Insight and recommendations

- Duplicate records are removed using table-appropriate keys.
- Missing cast character names are labelled `Unknown` so valid cast rows are retained.
- Release dates are converted to datetime.
- Zero budget/revenue values are preserved but excluded from financial calculations.
- Relationship tables are checked for orphan references.